In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np
import pycav

## Standing wave cavity

In [ ]:
cwl = 1.03e-6
aoi = np.deg2rad(5)

In [ ]:
cav = pycav.Cavity(1e-6, ring=False)
cav.add_mirror(r=0, aoi=0, idn="SESAM")
cav.add_space(d=1, idn="S_TD")
cav.add_mirror(r=1, aoi=aoi, idn="TD")
cav.add_space(d=0.2, idn="TD_R1")
cav.add_mirror(r=2, aoi=-aoi, idn="R1")
cav.add_space(d=0.3, idn="R1_OC")
cav.add_mirror(r=0, aoi=0, idn="OC")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 2))
d, w = cav.get_beam()
ax.plot(d, w * 1e6, label=["sagittal", "tangential"])
ax.vlines(cav.get_pos(), w.min() * 1e6, w.max() * 1e6, "k")
ax.set_xlabel("cavity length / m")
ax.set_ylabel("beam size / µm")
ax.legend()
fig.tight_layout()

In [ ]:
cav = pycav.Cavity(1e-6, 3e-2)
cav.add_mirror(r=0, aoi=0, idn="SESAM")
cav.add_space(d=1, idn="S_TD")
cav.add_mirror(r=1, aoi=aoi, idn="TD")
cav.add_space(d=0.2, idn="TD_R1")
cav.add_mirror(r=2, aoi=-aoi, idn="R1")
cav.add_space(d=0.3, idn="R1_OC")
cav.add_mirror(r=0, aoi=0, idn="OC")

In [ ]:
# vary length SESAM/disk, calculate beam size on disk, SESAM, and OC
vSD = np.linspace(0, 3, 1000)
vSD_dat = cav.vary_multi(("S_TD", "d", vSD), ["TD", "SESAM", "OC"])
# vary length disk/R1, calculate beam size on disk
vDR1 = np.linspace(0, 3, 1000)
vDR1_dat = cav.vary(("TD_R1", "d", vDR1), "TD")
# # vary length R1/OC, calculate beam size on disk
vR1O = np.linspace(0, 3, 1000)
vR1O_dat = cav.vary(("R1_OC", "d", vR1O), "TD")
# # vary radius of curvature of the disk, calculate beam size on disk
vrD = np.linspace(0.1, 2, 1000)
vrD_dat = cav.vary(("TD", "r", vrD), "TD")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3), ncols=4)
ax[0].plot(vSD, vSD_dat * 1e6)
ax[0].set_xlabel("length SESAM-disk / m")
ax[0].set_ylabel("beam size disk / µm")
ax[1].plot(vDR1, vDR1_dat * 1e6)
ax[1].set_xlabel("length disk-R1 / m")
ax[1].set_ylabel("beam size on disk / µm")
ax[2].plot(vR1O, vR1O_dat * 1e6)
ax[2].set_xlabel("length R1-OC / m")
ax[2].set_ylabel("beam size on disk / µm")
ax[3].plot(2 / vrD, vrD_dat * 1e6)
ax[3].set_xlabel("focal power of disk / (1/m)")
ax[3].set_ylabel("beam size on disk / µm")
fig.tight_layout()

In [ ]:
r0 = np.linspace(0, 1, 101)
aois = np.linspace(0, np.pi / 2, 46)
v0 = ("S_TD", "d", r0)
v1 = ("TD", "aoi", aois)

ws, wt = cav.vary_2D((v0, v1), "TD")

In [ ]:
fig, ax = plt.subplots(ncols=2)
cpls = ax[0].contourf(r0, aois, ws * 1e3, np.linspace(0, 1, 251))
cplt = ax[1].contourf(r0, aois, wt * 1e3, np.linspace(0, 1, 251))
ax[0].set_xlabel("variation: S_TD(d) / m")
ax[0].set_ylabel("variation: TD(aoi) / rad")
ax[1].set_xlabel("variation: S_TD(d) / m")
ax[1].set_ylabel("variation: TD(aoi) / rad")
cb = fig.colorbar(cpls)
cb.set_label("beam width / mm")
cb.set_ticks(np.linspace(0, 1, 11))
fig.tight_layout()

## Beam propagation

In [ ]:
w0 = [1e-3, 0.5e-3]

b = pycav.Beam(1e-6)
b.set_w0(w0)
b.add_space(d=1, idn="A")
b.add_lens(f=[8, 1.4], idn="L1")
b.add_space(d=1)
b.add_mirror(r=1, aoi=0)
b.add_space(1)
b.add_mirror(0.5, 0)
b.add_space(0.1)
b.add_bulk(0.5, 1.9, idn="YAG")
b.add_space(1)

# get beam
d, w = b.get_beam()
# vary length A, calculate beam size on L1
dv = np.linspace(0, 1, 11)
vd = b.vary(("A", "d", dv), "L1")

fig, ax = plt.subplots()
ax.plot(d, w, label=["sagittal", "tangential"])
ax.plot(dv, vd, "o", label=["sagittal on L1", "tangential on L1"])
ax.set_xlabel("length / m")
ax.set_ylabel("beam width / mm")
ax.legend()

## Ring cavity

In [ ]:
rc = pycav.Cavity(1e-6, ring=True)
rc.add_mirror(0, np.deg2rad(60))
rc.add_space(0.25)
rc.add_mirror(1, np.deg2rad(60))
rc.add_space(0.25)

d, w = rc.get_beam()

fig, ax = plt.subplots()
ax.plot(d, w * 1e3, label=["sagittal", "tangential"])
ax.plot(*rc.get_pos_plt(w.max() * 1e3), "k")
ax.set_xlabel("cavity length / m")
ax.set_ylabel("beam width / mm")
ax.legend()

### different initializers

In [ ]:
cav = pycav.Cavity(1e-6, ring=False)
cav.add_mirror(r=2, idn="SESAM")
cav.add_space(d=1, idn="S_TD")
cav.add_mirror(r=2, idn="TD")

d, w = cav.get_beam()

In [ ]:
b = pycav.Beam(1e-6)

b.set_q0(cav.get_qs()[0][0])
b.add_mirror(r=2, idn="SESAM")
b.add_space(d=1, idn="S_TD")

dd, ww = b.get_beam()

In [ ]:
bb = pycav.Beam(1e-6)

bb.set_w0R0(w[0, 0], -2)
bb.add_space(d=1, idn="S_TD")

ddd, www = bb.get_beam()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 2))

ax.plot(d, w * 1e6, label=["cavity mode", None])
ax.plot(dd, ww * 1e6, "r--", label=["set_q0()", None])
ax.plot(ddd, www * 1e6, "k:", label=["set_w0R0()", None])
ax.vlines(cav.get_pos(), w.min() * 1e6, w.max() * 1e6, "k")
ax.set_xlabel("cavity length / m")
ax.set_ylabel("beam size / µm")
ax.legend()
fig.tight_layout()

### Custom ABCD matrix

Elements without a dedicated `add_*()` method can be added with
`add_custom_section(Ms, Mt=None, n=None, idn=None)`, which takes the sagittal
and tangential ABCD matrices `[[A, B], [C, D]]`. The tangential matrix defaults
to the sagittal one. The section has no length.

As a check, an astigmatic thin lens given as custom matrices yields the same
beam as `add_lens()`:

In [ ]:
fs, ft = 0.5, 0.3
Ms = np.array(((1, 0), (-1 / fs, 1)))
Mt = np.array(((1, 0), (-1 / ft, 1)))

b = pycav.Beam(1e-6)
b.set_q0(cav.get_qs()[0][0])
b.add_mirror(r=2, idn="SESAM")
b.add_space(d=1, idn="S_L")
b.add_custom_section(Ms, Mt, idn="custom")
b.add_space(d=1, idn="L_E")
dd, ww = b.get_beam()

b2 = pycav.Beam(1e-6)
b2.set_q0(cav.get_qs()[0][0])
b2.add_mirror(r=2, idn="SESAM")
b2.add_space(d=1, idn="S_L")
b2.add_lens([fs, ft], idn="lens")
b2.add_space(d=1, idn="L_E")
dd2, ww2 = b2.get_beam()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 2))
ax.plot(dd2, ww2 * 1e6, "k", label=["add_lens()", None])
ax.plot(dd, ww * 1e6, "r--", label=["add_custom_section()", None])
ax.set_xlabel("propagation distance / m")
ax.set_ylabel("beam size / µm")
ax.legend()
fig.tight_layout()

In a standing-wave cavity, the beam passes each section in both directions.
The matrix for the reverse direction is calculated from the given matrix, so
asymmetric elements work, too. Here, a lens followed by a 0.2 m space is given
as a single custom matrix and compared with the same element built from
`add_lens()` and `add_space()`, while varying the distance to the first mirror:

In [ ]:
fs_c, ft_c = 0.4, 0.35
Ms_c = np.array(((1, 0), (-1 / fs_c, 1)))
Mt_c = np.array(((1, 0), (-1 / ft_c, 1)))


def make_cavity(custom):
    c = pycav.Cavity(1e-6)
    c.add_mirror(r=0, idn="M1")
    c.add_space(d=0.3, idn="M1_L")
    if custom:
        S = np.array(((1, 0.2), (0, 1)))
        c.add_custom_section(S @ Ms_c, S @ Mt_c, idn="element")
    else:
        c.add_lens([fs_c, ft_c], idn="lens")
        c.add_space(d=0.2)
    c.add_space(d=0.3)
    c.add_mirror(r=1, aoi=np.deg2rad(5), idn="M2")
    return c


dM1 = np.linspace(0, 1, 201)
w_ref = make_cavity(False).vary(("M1_L", "d", dM1), "M1")
w_cus = make_cavity(True).vary(("M1_L", "d", dM1), "M1")

fig, ax = plt.subplots(figsize=(9, 2))
ax.plot(dM1, w_ref * 1e6, "k", label=["add_lens() + add_space()", None])
ax.plot(dM1, w_cus * 1e6, "r--", label=["add_custom_section()", None])
ax.set_xlabel("distance M1-lens / m")
ax.set_ylabel("beam size on M1 / µm")
ax.legend()
fig.tight_layout()